# Notebook 04: Feature Engineering & Preprocessing Pipeline

## Credit Card Customer Churn & Segmentation
**Objective:** Construct interpretable behavioral features with zero data leakage, inspect multicollinearity, and build a reusable Scikit-learn `ColumnTransformer` and `Pipeline`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_raw_data
from src.features.build_features import FeatureEngineer, add_engineered_features
from src.data.preprocess import split_data, create_preprocessor, ALL_NUMERICAL_FEATURES, CATEGORICAL_FEATURES
from src.utils.helpers import TARGET_COLUMN, ID_COLUMN

df = load_raw_data()
X_train, X_test, y_train, y_test = split_data(df)
print(f"Train rows: {len(X_train):,}, Holdout test rows: {len(X_test):,}")

## 1. Feature Derivations & Rationale
We construct features with explicit business interpretability:
1. `avg_trans_value`: $\frac{\text{Total\_Trans\_Amt}}{\text{Total\_Trans\_Ct}}$ (average ticket size per transaction)
2. `trans_per_month`: $\frac{\text{Total\_Trans\_Ct}}{\text{Months\_on\_book}}$ (tenure-normalized swipe frequency)
3. `trans_amt_per_month`: $\frac{\text{Total\_Trans\_Amt}}{\text{Months\_on\_book}}$ (tenure-normalized spend velocity)
4. `inactivity_ratio`: $\frac{\text{Months\_Inactive\_12\_mon}}{12.0}$ (proportion of year card was dormant)
5. `declining_trans_ct_flag`: Binary indicator for severe drop in transaction count ($< 0.60$)
6. `declining_trans_amt_flag`: Binary indicator for severe drop in spend ($< 0.60$)

In [ ]:
fe = FeatureEngineer()
X_train_fe = fe.fit_transform(X_train)
engineered_cols = ["avg_trans_value", "trans_per_month", "trans_amt_per_month", "inactivity_ratio", "declining_trans_ct_flag", "declining_trans_amt_flag"]
X_train_fe[engineered_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]

## 2. Multicollinearity & Correlation Analysis
We examine pairwise correlations among numerical predictors.

In [ ]:
corr = X_train_fe[ALL_NUMERICAL_FEATURES].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap of Numerical Features (Train Set Only)")
plt.tight_layout()
plt.show()

## 3. Scikit-learn Pipeline Assembly
We assemble the `ColumnTransformer`:
- Numerical Pipeline: Median Imputer + `StandardScaler`
- Categorical Pipeline: Constant Imputer + `OneHotEncoder(drop='first', handle_unknown='ignore')`
- Transformers are fitted strictly on `X_train` to eliminate data leakage.

In [ ]:
preprocessor = create_preprocessor()
preprocessor.fit(X_train_fe)
X_train_transformed = preprocessor.transform(X_train_fe)
print("Transformed training matrix shape:", X_train_transformed.shape)